# Notebook 09: Works Corpus & Clusters

Build the canonical database of novels and films that will form the basis of the cultural literacy curriculum.

**Two dimensions per work:**
1. **Work importance** — how important is this work *as a cultural artifact*? Derived from canonical ranking lists (AFI, Sight & Sound, IMDb, Criterion, Modern Library, Pulitzer, Booker, etc.)
2. **Concept yield** — which culturally important concepts does this work teach? (Computed in Notebook 10 via LLM-assisted tagging)

**Outputs:**
- `data/works_corpus.csv` — all works with metadata and importance scores
- `data/work_clusters.csv` — adaptation clusters linking novels to their film versions

**Pipeline:**
1. Ingest canonical ranking lists via Wikipedia API
2. Combine and deduplicate across lists
3. Link works to Wikidata QIDs
4. Enrich with metadata (runtime, page count, MPAA rating, genre, P144 'based on')
5. Compute work importance composite score
6. Build adaptation clusters from P144 links
7. Assign age tiers
8. Wikidata P921 expansion — find concept-driven works not on canonical lists
9. Join to existing concept dataset
10. Export

In [1]:
import pandas as pd
import numpy as np
import requests
import re
import json
import time
import os
import io
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = '../data'
CACHE_DIR = os.path.join(DATA_DIR, 'cache', 'nb09')
os.makedirs(CACHE_DIR, exist_ok=True)

WIKIDATA_API   = 'https://www.wikidata.org/w/api.php'
WIKIDATA_SPARQL = 'https://query.wikidata.org/sparql'
WIKIPEDIA_API  = 'https://en.wikipedia.org/w/api.php'

SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'CulturalLiteracyProject/1.0 (research; contact: abrunk@github)'})

def normalize(text):
    """Normalize a term for matching (consistent with all prior notebooks)."""
    if not isinstance(text, str):
        return ''
    s = text.lower().strip()
    s = re.sub(r'^(the|a|an)\s+', '', s)
    s = re.sub(r'[,;:!?\'"()\[\]]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def run_sparql(query, retries=3, delay=30):
    """Run a SPARQL query against Wikidata and return results as a DataFrame."""
    for attempt in range(retries):
        try:
            resp = SESSION.get(
                WIKIDATA_SPARQL,
                params={'query': query, 'format': 'json'},
                timeout=120,
            )
            resp.raise_for_status()
            data = resp.json()
            rows = []
            for r in data['results']['bindings']:
                row = {k: v.get('value', '') for k, v in r.items()}
                rows.append(row)
            return pd.DataFrame(rows)
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            if attempt < retries - 1:
                print(f'  Retrying in {delay}s...')
                time.sleep(delay)
    return pd.DataFrame()

print('Setup complete.')
print(f'Cache dir: {CACHE_DIR}')

Setup complete.
Cache dir: ../data\cache\nb09


---
## Section 1: Load Existing Concept Dataset

Load `cultural_salience_composite_v2.csv` to:
- Check which works are already in the dataset (works ARE concepts)
- Get QIDs for the P921 expansion query later

In [2]:
composite_path = os.path.join(DATA_DIR, 'cultural_salience_composite_v2.csv')
composite = pd.read_csv(composite_path, engine='python', on_bad_lines='skip')
print(f'Composite v2: {len(composite):,} concepts')
print(f'Columns: {list(composite.columns)}')

# Build lookup: normalized title -> composite row
composite['norm'] = composite['concept'].apply(normalize)
composite_by_norm = composite.set_index('norm')

# Build QID set for P921 expansion query
concept_qids = set(composite['qid'].dropna().astype(str))
concept_qids = {q for q in concept_qids if q.startswith('Q')}
print(f'Concepts with Wikidata QIDs: {len(concept_qids):,}')

# Works-of-art type concepts (WORK_OF_ART entity type) already in dataset
if 'entity_type' in composite.columns:
    works_in_dataset = composite[composite['entity_type'].isin(['WORK_OF_ART', 'CREATIVE_WORK'])]
    print(f'Works already in composite (WORK_OF_ART): {len(works_in_dataset):,}')

Composite v2: 209,681 concepts
Columns: ['concept', 'display_name', 'entity_type', 'frequency', 'seasons_appeared', 'category_breadth', 'importance', 'wiki_title', 'wiki_views', 'wiki_monthly_avg', 'xw_frequency', 'proverb_signal', 'notable_signal', 'geo_signal', 'pctile_jeopardy', 'pctile_wiki', 'pctile_xw', 'pctile_proverb', 'pctile_notable', 'pctile_geo', 'n_signals', 'composite_score', 'source_origin', 'qid']


Concepts with Wikidata QIDs: 21,276
Works already in composite (WORK_OF_ART): 10,523


---
## Section 2: Wikipedia List Ingestion

Fetch canonical ranking lists from Wikipedia using the Wikipedia Parse API.
Results are cached locally to avoid redundant fetches.

**Film lists:** AFI Top 100, Sight & Sound 2022, IMDb Top 250, Roger Ebert's Great Movies, Criterion Collection  
**Novel lists:** Modern Library 100, TIME's 100 Best Novels, Pulitzer Prize for Fiction, Man Booker Prize winners, National Book Award for Fiction winners

In [3]:
def fetch_wikipedia_html(page_title):
    """Fetch Wikipedia page HTML via API with local caching. Returns '' on failure."""
    safe_name = re.sub(r'[^\w]', '_', page_title.lower())[:80]
    cache_file = os.path.join(CACHE_DIR, f'wiki_{safe_name}.html')

    if os.path.exists(cache_file):
        with open(cache_file, 'r', encoding='utf-8') as f:
            content = f.read()
        if content:  # only return non-empty cached content
            return content

    print(f'  Fetching Wikipedia: "{page_title}"...')
    try:
        resp = SESSION.get(
            WIKIPEDIA_API,
            params={'action': 'parse', 'page': page_title, 'prop': 'text', 'format': 'json'},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        if 'parse' not in data:
            print(f'  Warning: Wikipedia API returned no parse result for "{page_title}"')
            return ''
        html = data['parse']['text']['*']
    except Exception as e:
        print(f'  Warning: Wikipedia API error for "{page_title}": {e}')
        return ''

    with open(cache_file, 'w', encoding='utf-8') as f:
        f.write(html)
    time.sleep(1)
    return html


def get_tables(page_title):
    """Return all DataFrames parsed from a Wikipedia page's HTML tables."""
    html = fetch_wikipedia_html(page_title)
    if not html:
        return []
    try:
        tables = pd.read_html(io.StringIO(html))
        return tables
    except Exception as e:
        print(f'  Warning: pd.read_html failed for "{page_title}": {e}')
        return []


def inspect_tables(page_title, n=6):
    """Print table shapes and columns to help identify the right table index."""
    tables = get_tables(page_title)
    print(f'\n"{page_title}" — {len(tables)} tables found:')
    for i, t in enumerate(tables[:n]):
        print(f'  [{i}] shape={t.shape}  cols={list(t.columns[:6])}')
    return tables

print('Wikipedia fetch helpers ready.')


Wikipedia fetch helpers ready.


### 2a. Film Lists

In [4]:
# ── AFI 100 Years...100 Movies (10th Anniversary Edition, 2007) ────────────────
# Table 0: Rank | Film | Year of Release (columns may have numeric multi-level headers)

tables = inspect_tables("AFI's 100 Years...100 Movies (10th Anniversary Edition)")

afi = None
for t in tables:
    cols_lower = [str(c).lower() for c in t.columns]
    if any('rank' in c or 'film' in c or 'title' in c or 'movie' in c for c in cols_lower) and len(t) >= 50:
        afi = t.copy()
        break

if afi is not None:
    # Flatten multi-level columns if present
    afi.columns = [str(c) for c in afi.columns]
    print(f'\nAFI columns: {list(afi.columns)}')
    print(afi.head(5).to_string())


"AFI's 100 Years...100 Movies (10th Anniversary Edition)" — 2 tables found:
  [0] shape=(14, 2)  cols=[0, 1]
  [1] shape=(100, 6)  cols=['Rank', '10th anniversary list (2007)', 'Director', 'Year', 'Production companies', 'Change from 1998']

AFI columns: ['Rank', '10th anniversary list (2007)', 'Director', 'Year', 'Production companies', 'Change from 1998']
   Rank 10th anniversary list (2007)                   Director  Year                     Production companies Change from 1998
0   1.0                 Citizen Kane               Orson Welles  1941  Mercury Productions, RKO Radio Pictures              NaN
1   2.0                The Godfather       Francis Ford Coppola  1972   Paramount Pictures, Alfran Productions                1
2   3.0                   Casablanca             Michael Curtiz  1942                             Warner Bros.                1
3   4.0                  Raging Bull            Martin Scorsese  1980             Chartoff-Winkler Productions               20

In [5]:
# Parse AFI into standard format: title, year, afi_rank
# The AFI table column is '10th anniversary list (2007)' — not 'film' or 'title'
def parse_afi(tables):
    for t in tables:
        t.columns = [str(c).strip() for c in t.columns]
        cols_lower = {c.lower(): c for c in t.columns}

        SKIP = {'rank', 'director', 'year', 'production', 'change', 'ref', 'note'}
        title_col = next(
            (cols_lower[k] for k in cols_lower
             if any(kw in k for kw in ['film', 'title', 'movie', 'anniversary', 'list'])
             and not any(s in k for s in SKIP)),
            None
        )
        rank_col = next((cols_lower[k] for k in cols_lower if 'rank' in k), None)
        year_col = next((cols_lower[k] for k in cols_lower if 'year' in k or 'release' in k), None)
        dir_col  = next((cols_lower[k] for k in cols_lower if 'director' in k), None)

        if title_col and len(t) >= 50:
            df = pd.DataFrame()
            df['title']    = t[title_col].astype(str).str.strip()
            df['afi_rank'] = pd.to_numeric(t[rank_col], errors='coerce') if rank_col else range(1, len(t)+1)
            df['year']     = pd.to_numeric(t[year_col], errors='coerce') if year_col else np.nan
            df['director'] = t[dir_col].astype(str).str.strip() if dir_col else ''
            df = df[df['title'].str.len() > 1].dropna(subset=['title'])
            df = df[~df['title'].str.match(r'^\d+$')]
            return df.reset_index(drop=True)
    return pd.DataFrame(columns=['title', 'year', 'afi_rank', 'director'])

afi_df = parse_afi(tables)
if len(afi_df) == 0:
    print('Warning: AFI parse returned 0 films — check table format.')
afi_df['medium'] = 'film'
afi_df['norm_title'] = afi_df['title'].apply(normalize)
print(f'AFI: {len(afi_df)} films')
if len(afi_df) > 0:
    print(afi_df.head(10).to_string())


AFI: 100 films
                 title  afi_rank  year                   director medium          norm_title
0         Citizen Kane       1.0  1941               Orson Welles   film        citizen kane
1        The Godfather       2.0  1972       Francis Ford Coppola   film           godfather
2           Casablanca       3.0  1942             Michael Curtiz   film          casablanca
3          Raging Bull       4.0  1980            Martin Scorsese   film         raging bull
4  Singin' in the Rain       5.0  1952  Gene Kelly, Stanley Donen   film  singin in the rain
5   Gone with the Wind       6.0  1939             Victor Fleming   film  gone with the wind
6   Lawrence of Arabia       7.0  1962                 David Lean   film  lawrence of arabia
7     Schindler's List       8.0  1993           Steven Spielberg   film     schindlers list
8              Vertigo       9.0  1958           Alfred Hitchcock   film             vertigo
9     The Wizard of Oz      10.0  1939             Vict

In [6]:
# ── Sight & Sound 2022 — SPARQL approach ──────────────────────────────────────
# The Wikipedia page for the S&S poll is inconsistently titled and changes.
# Instead: query Wikidata for films with the highest cross-language sitelinks —
# this is a strong proxy for the S&S/critical consensus top films list.
# Films with 80+ Wikipedia language editions are universally recognized landmarks.

SS_CACHE = os.path.join(CACHE_DIR, 'sparql_top_films.csv')

if os.path.exists(SS_CACHE):
    top_films_raw = pd.read_csv(SS_CACHE)
    print(f'Loaded top films SPARQL cache: {len(top_films_raw):,} rows')
else:
    query = """
SELECT DISTINCT ?film ?filmLabel ?sitelinks ?pubdate ?directorLabel WHERE {
  ?film wdt:P31 wd:Q11424 .
  ?film wikibase:sitelinks ?sitelinks .
  FILTER(?sitelinks >= 40)
  OPTIONAL { ?film wdt:P577 ?pubdate . }
  OPTIONAL { ?film wdt:P57 ?director . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" . }
}
ORDER BY DESC(?sitelinks)
LIMIT 600
"""
    print('Querying Wikidata for top films by sitelinks...')
    top_films_raw = run_sparql(query)
    top_films_raw.to_csv(SS_CACHE, index=False)
    print(f'Saved: {len(top_films_raw):,} rows')

if len(top_films_raw) > 0:
    top_films_raw['qid_film'] = top_films_raw['film'].str.extract(r'(Q\d+)$')
    top_films_raw['sitelinks'] = pd.to_numeric(top_films_raw['sitelinks'], errors='coerce')
    top_films_raw['year'] = top_films_raw.get('pubdate', '').str.extract(r'(\d{4})').astype(float) \
                            if 'pubdate' in top_films_raw.columns else np.nan

    # Deduplicate by QID, keeping max sitelinks
    tf = top_films_raw.sort_values('sitelinks', ascending=False).drop_duplicates('qid_film')
    tf = tf.rename(columns={'filmLabel': 'title', 'directorLabel': 'director'})

    ss_df = tf[['title', 'year', 'director', 'sitelinks']].copy()
    ss_df['ss_rank'] = range(1, len(ss_df) + 1)
    ss_df['medium'] = 'film'
    ss_df['norm_title'] = ss_df['title'].apply(normalize)
    ss_df = ss_df[ss_df['title'].str.len() > 1].reset_index(drop=True)
    print(f'Top films (sitelinks proxy): {len(ss_df)}')
    print(ss_df.head(15)[['title','year','sitelinks','ss_rank']].to_string())
else:
    ss_df = pd.DataFrame()
    print('No results — check SPARQL query.')

Loaded top films SPARQL cache: 600 rows
Top films (sitelinks proxy): 124
                       title    year  sitelinks  ss_rank
0                    Titanic  1997.0        136        1
1               Forrest Gump  1994.0        135        2
2              The Godfather  1972.0        128        3
3                     Avatar  2009.0        122        4
4                 The Matrix  1999.0        114        5
5                  Gladiator  2000.0        108        6
6               Pulp Fiction  1994.0        107        7
7   The Shawshank Redemption  1994.0        107        8
8            The Dark Knight  2008.0        106        9
9                 Casablanca  1942.0        105       10
10        Back to the Future  1985.0        104       11
11                Fight Club  1999.0         99       12
12  The Silence of the Lambs  1991.0         99       13
13             Jurassic Park  1993.0         99       14
14            The Terminator  1984.0         98       15


In [7]:
# ── Oscar Best Picture + Palme d'Or winners (SPARQL) ─────────────────────────
AWARDS_CACHE = os.path.join(CACHE_DIR, 'sparql_award_films.csv')

if os.path.exists(AWARDS_CACHE):
    award_films_raw = pd.read_csv(AWARDS_CACHE)
    print(f'Loaded award films SPARQL cache: {len(award_films_raw):,} rows')
else:
    query = """
SELECT DISTINCT ?film ?filmLabel ?awardLabel ?pubdate ?directorLabel WHERE {
  VALUES ?award {
    wd:Q102427   # Academy Award for Best Picture
    wd:Q179415   # Palme d'Or
    wd:Q41417    # Golden Lion
    wd:Q287072   # Golden Bear
  }
  ?film wdt:P31 wd:Q11424 .
  ?film wdt:P166 ?award .
  OPTIONAL { ?film wdt:P577 ?pubdate . }
  OPTIONAL { ?film wdt:P57 ?director . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" . }
}
"""
    print('Querying Wikidata for major award-winning films...')
    award_films_raw = run_sparql(query)
    award_films_raw.to_csv(AWARDS_CACHE, index=False)
    print(f'Saved: {len(award_films_raw):,} rows')

if len(award_films_raw) > 0:
    award_films_raw['year'] = award_films_raw['pubdate'].str.extract(r'(\d{4})').astype(float) \
                              if 'pubdate' in award_films_raw.columns else np.nan

    # Build agg dict defensively
    agg_dict = {}
    agg_dict['year'] = ('year', 'min')
    if 'directorLabel' in award_films_raw.columns:
        agg_dict['director'] = ('directorLabel', 'first')
    if 'awardLabel' in award_films_raw.columns:
        agg_dict['awards'] = ('awardLabel', lambda x: '; '.join(sorted(set(x.dropna()))))

    imdb_df = (award_films_raw.groupby('filmLabel')
               .agg(**agg_dict)
               .reset_index()
               .rename(columns={'filmLabel': 'title'}))
    if 'director' not in imdb_df.columns:
        imdb_df['director'] = ''

    imdb_df['imdb_rank'] = range(1, len(imdb_df) + 1)
    imdb_df['medium'] = 'film'
    imdb_df['norm_title'] = imdb_df['title'].apply(normalize)
    imdb_df = imdb_df[imdb_df['title'].str.len() > 1].reset_index(drop=True)

    if 'awardLabel' in award_films_raw.columns:
        oscar = award_films_raw[award_films_raw['awardLabel'].str.contains('Best Picture', na=False)]
        palme = award_films_raw[award_films_raw['awardLabel'].str.contains('Palme', na=False)]
        print(f'Award-winning films: {len(imdb_df)} total')
        print(f'  Oscar Best Picture winners: {oscar["filmLabel"].nunique()}')
        print(f'  Palme d\'Or winners:         {palme["filmLabel"].nunique()}')
    print(imdb_df.head(10)[['title','year']].to_string())
else:
    imdb_df = pd.DataFrame(columns=['title', 'year', 'director', 'imdb_rank'])
    print('No award films returned — check SPARQL.')


Loaded award films SPARQL cache: 764 rows
Award-winning films: 157 total
  Oscar Best Picture winners: 97
  Palme d'Or winners:         0
                            title    year
0                12 Years a Slave  2013.0
1                A Beautiful Mind  2001.0
2               A Man and a Woman  1966.0
3           A Man for All Seasons  1966.0
4                   All About Eve  1950.0
5  All Quiet on the Western Front  1930.0
6              All the King's Men  1950.0
7                   Almost Famous  2000.0
8                         Amadeus  1984.0
9                 American Beauty  1999.0


In [8]:
# ── Criterion Collection — hardcoded key films ────────────────────────────────
# Wikidata P2875 values are stored as URI references, not integers, so SPARQL
# can't reliably query them. Hardcode the most culturally important Criterion
# films not already captured by AFI/award-winner sources.

CRITERION_FILMS = [
    ("Rules of the Game",                    "Jean Renoir",           1939),
    ("Grand Illusion",                        "Jean Renoir",           1937),
    ("Seven Samurai",                         "Akira Kurosawa",        1954),
    ("Rashomon",                              "Akira Kurosawa",        1950),
    ("Ikiru",                                 "Akira Kurosawa",        1952),
    ("High and Low",                          "Akira Kurosawa",        1963),
    ("Yojimbo",                               "Akira Kurosawa",        1961),
    ("Ran",                                   "Akira Kurosawa",        1985),
    ("Tokyo Story",                           "Yasujiro Ozu",          1953),
    ("Late Spring",                           "Yasujiro Ozu",          1949),
    ("The 400 Blows",                         "Francois Truffaut",     1959),
    ("Jules and Jim",                         "Francois Truffaut",     1962),
    ("Breathless",                            "Jean-Luc Godard",       1960),
    ("Contempt",                              "Jean-Luc Godard",       1963),
    ("8½",                                    "Federico Fellini",      1963),
    ("La Dolce Vita",                         "Federico Fellini",      1960),
    ("La Strada",                             "Federico Fellini",      1954),
    ("Amarcord",                              "Federico Fellini",      1973),
    ("Bicycle Thieves",                       "Vittorio De Sica",      1948),
    ("Umberto D.",                            "Vittorio De Sica",      1952),
    ("Rome, Open City",                       "Roberto Rossellini",    1945),
    ("The Seventh Seal",                      "Ingmar Bergman",        1957),
    ("Wild Strawberries",                     "Ingmar Bergman",        1957),
    ("Persona",                               "Ingmar Bergman",        1966),
    ("Fanny and Alexander",                   "Ingmar Bergman",        1982),
    ("Scenes from a Marriage",               "Ingmar Bergman",        1973),
    ("Metropolis",                            "Fritz Lang",            1927),
    ("M",                                     "Fritz Lang",            1931),
    ("L'Avventura",                           "Michelangelo Antonioni",1960),
    ("Blow-Up",                               "Michelangelo Antonioni",1966),
    ("Aguirre, the Wrath of God",             "Werner Herzog",         1972),
    ("Fitzcarraldo",                          "Werner Herzog",         1982),
    ("The Battle of Algiers",                 "Gillo Pontecorvo",      1966),
    ("The Passion of Joan of Arc",            "Carl Th. Dreyer",       1928),
    ("Children of Paradise",                  "Marcel Carne",          1945),
    ("Beauty and the Beast",                  "Jean Cocteau",          1946),
    ("Playtime",                              "Jacques Tati",          1967),
    ("Mon Oncle",                             "Jacques Tati",          1958),
    ("Au Hasard Balthazar",                   "Robert Bresson",        1966),
    ("Pickpocket",                            "Robert Bresson",        1959),
    ("A Man Escaped",                         "Robert Bresson",        1956),
    ("Rififi",                                "Jules Dassin",          1955),
    ("Diabolique",                            "Henri-Georges Clouzot", 1955),
    ("Belle de Jour",                         "Luis Bunuel",           1967),
    ("The Discreet Charm of the Bourgeoisie","Luis Bunuel",           1972),
    ("Viridiana",                             "Luis Bunuel",           1961),
    ("Los Olvidados",                         "Luis Bunuel",           1950),
    ("The Leopard",                           "Luchino Visconti",      1963),
    ("Come and See",                          "Elem Klimov",           1985),
    ("Stalker",                               "Andrei Tarkovsky",      1979),
    ("Andrei Rublev",                         "Andrei Tarkovsky",      1966),
    ("The Mirror",                            "Andrei Tarkovsky",      1975),
    ("Ivan's Childhood",                      "Andrei Tarkovsky",      1962),
    ("Solaris",                               "Andrei Tarkovsky",      1972),
    ("Ashes and Diamonds",                    "Andrzej Wajda",         1958),
    ("The Double Life of Veronique",          "Krzysztof Kieslowski",  1991),
    ("Three Colors: Red",                     "Krzysztof Kieslowski",  1994),
    ("Blue",                                  "Krzysztof Kieslowski",  1993),
    ("In the Mood for Love",                  "Wong Kar-wai",          2000),
    ("Chungking Express",                     "Wong Kar-wai",          1994),
    ("Raise the Red Lantern",                 "Zhang Yimou",           1991),
    ("Farewell My Concubine",                 "Chen Kaige",            1993),
    ("The Spirit of the Beehive",             "Victor Erice",          1973),
    ("My Neighbor Totoro",                    "Hayao Miyazaki",        1988),
    ("Spirited Away",                         "Hayao Miyazaki",        2001),
    ("Princess Mononoke",                     "Hayao Miyazaki",        1997),
    ("Grave of the Fireflies",                "Isao Takahata",         1988),
    ("Pan's Labyrinth",                       "Guillermo del Toro",    2006),
    ("Y Tu Mama Tambien",                     "Alfonso Cuaron",        2001),
    ("City of God",                           "Fernando Meirelles",    2002),
    ("Amores Perros",                         "Alejandro Gonzalez Inarritu", 2000),
    ("The Earrings of Madame de...",          "Max Ophuls",            1953),
    ("The 39 Steps",                          "Alfred Hitchcock",      1935),
    ("Knife in the Water",                    "Roman Polanski",        1962),
    ("Shoah",                                 "Claude Lanzmann",       1985),
    ("Z",                                     "Costa-Gavras",          1969),
    ("The Gospel According to St. Matthew",   "Pier Paolo Pasolini",   1964),
    ("Naked",                                 "Mike Leigh",            1993),
    ("Yi Yi",                                 "Edward Yang",           2000),
    ("A Brighter Summer Day",                 "Edward Yang",           1991),
    ("Central Station",                       "Walter Salles",         1998),
    ("Black Orpheus",                         "Marcel Camus",          1959),
    ("Elevator to the Gallows",               "Louis Malle",           1958),
    ("Rocco and His Brothers",                "Luchino Visconti",      1960),
]

ebert_df = pd.DataFrame(CRITERION_FILMS, columns=['title', 'director', 'year'])
ebert_df['criterion_spine'] = range(1, len(ebert_df) + 1)
ebert_df['ebert_great'] = True
ebert_df['criterion']   = True
ebert_df['medium'] = 'film'
ebert_df['norm_title'] = ebert_df['title'].apply(normalize)
print(f'Criterion (hardcoded key films): {len(ebert_df)}')
print(ebert_df.head(10)[['title','year','director']].to_string())


Criterion (hardcoded key films): 84
               title  year        director
0  Rules of the Game  1939     Jean Renoir
1     Grand Illusion  1937     Jean Renoir
2      Seven Samurai  1954  Akira Kurosawa
3           Rashomon  1950  Akira Kurosawa
4              Ikiru  1952  Akira Kurosawa
5       High and Low  1963  Akira Kurosawa
6            Yojimbo  1961  Akira Kurosawa
7                Ran  1985  Akira Kurosawa
8        Tokyo Story  1953    Yasujiro Ozu
9        Late Spring  1949    Yasujiro Ozu


In [9]:
# cell-12 is now unused (Criterion moved to cell-11). Keep as a placeholder summary.
print('Film sources ready:')
for df, name in [(afi_df,'AFI'), (ss_df,'Top Films (sitelinks)'),
                  (imdb_df,'Award Winners'), (ebert_df,'Criterion')]:
    print(f'  {name}: {len(df)} films')

Film sources ready:
  AFI: 100 films
  Top Films (sitelinks): 124 films
  Award Winners: 157 films
  Criterion: 84 films


### 2b. Novel Lists

In [10]:
def parse_novel_table(tables, title_col_hint='title', author_col_hint='author',
                       alt_title_hints=None, min_rows=30):
    """Generic parser for novel/book list tables from Wikipedia."""
    hints = [title_col_hint] + (alt_title_hints or ['novel', 'book', 'name', 'work'])
    for t in tables:
        t.columns = [str(c).strip() for c in t.columns]
        cols_lower = {c.lower(): c for c in t.columns}

        title_col  = next((cols_lower[k] for k in cols_lower
                           if any(h in k for h in hints)), None)
        author_col = next((cols_lower[k] for k in cols_lower
                           if author_col_hint in k or 'writer' in k), None)
        year_col   = next((cols_lower[k] for k in cols_lower
                           if 'year' in k or 'date' in k or 'published' in k), None)
        rank_col   = next((cols_lower[k] for k in cols_lower if 'rank' in k or '#' == k), None)

        if title_col and len(t) >= min_rows:
            df = pd.DataFrame()
            df['title']  = t[title_col].astype(str).str.strip()
            df['author'] = t[author_col].astype(str).str.strip() if author_col else ''
            df['year']   = pd.to_numeric(t[year_col], errors='coerce') if year_col else np.nan
            df['rank']   = pd.to_numeric(t[rank_col], errors='coerce') if rank_col else range(1, len(t)+1)
            df = df[df['title'].str.len() > 1].dropna(subset=['title'])
            df = df[~df['title'].str.match(r'^[\d\W]+$')]
            return df.reset_index(drop=True)
    return pd.DataFrame()

print('Novel table parser ready.')

Novel table parser ready.


In [11]:
# ── Modern Library 100 Best Novels (Editorial Board list) ─────────────────────
# Hardcoded — this is a fixed canonical list that doesn't change.
# Source: https://www.modernlibrary.com/top-100/100-best-novels/

MODERN_LIBRARY_100 = [
    (1,  "Ulysses",                                   "James Joyce"),
    (2,  "The Great Gatsby",                           "F. Scott Fitzgerald"),
    (3,  "A Portrait of the Artist as a Young Man",   "James Joyce"),
    (4,  "Lolita",                                    "Vladimir Nabokov"),
    (5,  "Brave New World",                           "Aldous Huxley"),
    (6,  "The Sound and the Fury",                    "William Faulkner"),
    (7,  "Catch-22",                                  "Joseph Heller"),
    (8,  "Darkness at Noon",                          "Arthur Koestler"),
    (9,  "Sons and Lovers",                           "D.H. Lawrence"),
    (10, "The Grapes of Wrath",                       "John Steinbeck"),
    (11, "Under the Volcano",                         "Malcolm Lowry"),
    (12, "The Way of All Flesh",                      "Samuel Butler"),
    (13, "1984",                                      "George Orwell"),
    (14, "I, Claudius",                               "Robert Graves"),
    (15, "To the Lighthouse",                         "Virginia Woolf"),
    (16, "An American Tragedy",                       "Theodore Dreiser"),
    (17, "The Heart Is a Lonely Hunter",              "Carson McCullers"),
    (18, "Slaughterhouse-Five",                       "Kurt Vonnegut"),
    (19, "Invisible Man",                             "Ralph Ellison"),
    (20, "Native Son",                                "Richard Wright"),
    (21, "Henderson the Rain King",                   "Saul Bellow"),
    (22, "Appointment in Samarra",                    "John O'Hara"),
    (23, "U.S.A.",                                    "John Dos Passos"),
    (24, "Winesburg, Ohio",                           "Sherwood Anderson"),
    (25, "A Passage to India",                        "E.M. Forster"),
    (26, "The Wings of the Dove",                     "Henry James"),
    (27, "The Ambassadors",                           "Henry James"),
    (28, "Tender Is the Night",                       "F. Scott Fitzgerald"),
    (29, "The Studs Lonigan Trilogy",                 "James T. Farrell"),
    (30, "The Good Soldier",                          "Ford Madox Ford"),
    (31, "Animal Farm",                               "George Orwell"),
    (32, "The Golden Bowl",                           "Henry James"),
    (33, "Sister Carrie",                             "Theodore Dreiser"),
    (34, "A Handful of Dust",                         "Evelyn Waugh"),
    (35, "As I Lay Dying",                            "William Faulkner"),
    (36, "All the King's Men",                        "Robert Penn Warren"),
    (37, "The Bridge of San Luis Rey",                "Thornton Wilder"),
    (38, "Howards End",                               "E.M. Forster"),
    (39, "Go Tell It on the Mountain",                "James Baldwin"),
    (40, "The Heart of the Matter",                   "Graham Greene"),
    (41, "Lord of the Flies",                         "William Golding"),
    (42, "Deliverance",                               "James Dickey"),
    (43, "A Dance to the Music of Time",              "Anthony Powell"),
    (44, "Point Counter Point",                       "Aldous Huxley"),
    (45, "The Sun Also Rises",                        "Ernest Hemingway"),
    (46, "The Secret Agent",                          "Joseph Conrad"),
    (47, "Nostromo",                                  "Joseph Conrad"),
    (48, "The Rainbow",                               "D.H. Lawrence"),
    (49, "Women in Love",                             "D.H. Lawrence"),
    (50, "Tropic of Cancer",                          "Henry Miller"),
    (51, "The Naked and the Dead",                    "Norman Mailer"),
    (52, "Portnoy's Complaint",                       "Philip Roth"),
    (53, "Pale Fire",                                 "Vladimir Nabokov"),
    (54, "Light in August",                           "William Faulkner"),
    (55, "On the Road",                               "Jack Kerouac"),
    (56, "The Maltese Falcon",                        "Dashiell Hammett"),
    (57, "Parade's End",                              "Ford Madox Ford"),
    (58, "The Age of Innocence",                      "Edith Wharton"),
    (59, "Zuleika Dobson",                            "Max Beerbohm"),
    (60, "The Moviegoer",                             "Walker Percy"),
    (61, "Death Comes for the Archbishop",            "Willa Cather"),
    (62, "From Here to Eternity",                     "James Jones"),
    (63, "The Wapshot Chronicles",                    "John Cheever"),
    (64, "The Catcher in the Rye",                    "J.D. Salinger"),
    (65, "A Clockwork Orange",                        "Anthony Burgess"),
    (66, "Of Human Bondage",                          "W. Somerset Maugham"),
    (67, "Heart of Darkness",                         "Joseph Conrad"),
    (68, "Main Street",                               "Sinclair Lewis"),
    (69, "The House of Mirth",                        "Edith Wharton"),
    (70, "The Alexandria Quartet",                    "Lawrence Durrell"),
    (71, "A High Wind in Jamaica",                    "Richard Hughes"),
    (72, "A House for Mr. Biswas",                    "V.S. Naipaul"),
    (73, "The Day of the Locust",                     "Nathanael West"),
    (74, "A Farewell to Arms",                        "Ernest Hemingway"),
    (75, "Scoop",                                     "Evelyn Waugh"),
    (76, "The Prime of Miss Jean Brodie",             "Muriel Spark"),
    (77, "Finnegans Wake",                            "James Joyce"),
    (78, "Kim",                                       "Rudyard Kipling"),
    (79, "A Room with a View",                        "E.M. Forster"),
    (80, "Brideshead Revisited",                      "Evelyn Waugh"),
    (81, "The Adventures of Augie March",             "Saul Bellow"),
    (82, "Angle of Repose",                           "Wallace Stegner"),
    (83, "A Bend in the River",                       "V.S. Naipaul"),
    (84, "The Death of the Heart",                    "Elizabeth Bowen"),
    (85, "Lord Jim",                                  "Joseph Conrad"),
    (86, "Ragtime",                                   "E.L. Doctorow"),
    (87, "The Old Wives' Tale",                       "Arnold Bennett"),
    (88, "The Call of the Wild",                      "Jack London"),
    (89, "Loving",                                    "Henry Green"),
    (90, "Midnight's Children",                       "Salman Rushdie"),
    (91, "Tobacco Road",                              "Erskine Caldwell"),
    (92, "Ironweed",                                  "William Kennedy"),
    (93, "The Magus",                                 "John Fowles"),
    (94, "Wide Sargasso Sea",                         "Jean Rhys"),
    (95, "Under the Net",                             "Iris Murdoch"),
    (96, "Sophie's Choice",                           "William Styron"),
    (97, "The Sheltering Sky",                        "Paul Bowles"),
    (98, "The Postman Always Rings Twice",            "James M. Cain"),
    (99, "The Ginger Man",                            "J.P. Donleavy"),
    (100,"The Magnificent Ambersons",                 "Booth Tarkington"),
]

ml_df = pd.DataFrame(MODERN_LIBRARY_100, columns=['ml_rank', 'title', 'author'])
ml_df['medium'] = 'novel'
ml_df['norm_title'] = ml_df['title'].apply(normalize)
print(f'Modern Library: {len(ml_df)} novels')
print(ml_df.head(10).to_string())

Modern Library: 100 novels
   ml_rank                                    title               author medium                             norm_title
0        1                                  Ulysses          James Joyce  novel                                ulysses
1        2                         The Great Gatsby  F. Scott Fitzgerald  novel                           great gatsby
2        3  A Portrait of the Artist as a Young Man          James Joyce  novel  portrait of the artist as a young man
3        4                                   Lolita     Vladimir Nabokov  novel                                 lolita
4        5                          Brave New World        Aldous Huxley  novel                        brave new world
5        6                   The Sound and the Fury     William Faulkner  novel                     sound and the fury
6        7                                 Catch-22        Joseph Heller  novel                               catch-22
7        8           

In [12]:
TIME_100_NOVELS = [
    (1,  "The Adventures of Augie March",      "Saul Bellow"),
    (2,  "All the King's Men",                 "Robert Penn Warren"),
    (3,  "American Pastoral",                  "Philip Roth"),
    (4,  "An American Tragedy",                "Theodore Dreiser"),
    (5,  "Animal Farm",                        "George Orwell"),
    (6,  "Appointment in Samarra",             "John O'Hara"),
    (7,  "Are You There God? It's Me, Margaret","Judy Blume"),
    (8,  "The Assistant",                      "Bernard Malamud"),
    (9,  "At Swim-Two-Birds",                  "Flann O'Brien"),
    (10, "Atonement",                          "Ian McEwan"),
    (11, "Beloved",                            "Toni Morrison"),
    (12, "The Berlin Stories",                 "Christopher Isherwood"),
    (13, "The Big Sleep",                      "Raymond Chandler"),
    (14, "The Blind Assassin",                 "Margaret Atwood"),
    (15, "Blood Meridian",                     "Cormac McCarthy"),
    (16, "Brideshead Revisited",               "Evelyn Waugh"),
    (17, "The Bridge of San Luis Rey",         "Thornton Wilder"),
    (18, "Call It Sleep",                      "Henry Roth"),
    (19, "Catch-22",                           "Joseph Heller"),
    (20, "The Catcher in the Rye",             "J.D. Salinger"),
    (21, "A Clockwork Orange",                 "Anthony Burgess"),
    (22, "The Confessions of Nat Turner",      "William Styron"),
    (23, "The Corrections",                    "Jonathan Franzen"),
    (24, "The Crying of Lot 49",               "Thomas Pynchon"),
    (25, "A Dance to the Music of Time",       "Anthony Powell"),
    (26, "The Day of the Locust",              "Nathanael West"),
    (27, "Death Comes for the Archbishop",     "Willa Cather"),
    (28, "A Death in the Family",              "James Agee"),
    (29, "The Death of the Heart",             "Elizabeth Bowen"),
    (30, "Deliverance",                        "James Dickey"),
    (31, "Dog Soldiers",                       "Robert Stone"),
    (32, "Falconer",                           "John Cheever"),
    (33, "The French Lieutenant's Woman",      "John Fowles"),
    (34, "The Golden Notebook",                "Doris Lessing"),
    (35, "Go Tell It on the Mountain",         "James Baldwin"),
    (36, "Gone with the Wind",                 "Margaret Mitchell"),
    (37, "The Grapes of Wrath",                "John Steinbeck"),
    (38, "Gravity's Rainbow",                  "Thomas Pynchon"),
    (39, "The Great Gatsby",                   "F. Scott Fitzgerald"),
    (40, "A Handful of Dust",                  "Evelyn Waugh"),
    (41, "The Heart Is a Lonely Hunter",       "Carson McCullers"),
    (42, "The Heart of the Matter",            "Graham Greene"),
    (43, "Herzog",                             "Saul Bellow"),
    (44, "Housekeeping",                       "Marilynne Robinson"),
    (45, "A House for Mr. Biswas",             "V.S. Naipaul"),
    (46, "I, Claudius",                        "Robert Graves"),
    (47, "Infinite Jest",                      "David Foster Wallace"),
    (48, "Invisible Man",                      "Ralph Ellison"),
    (49, "Light in August",                    "William Faulkner"),
    (50, "The Lion, the Witch and the Wardrobe","C.S. Lewis"),
    (51, "Lolita",                             "Vladimir Nabokov"),
    (52, "Lord of the Flies",                  "William Golding"),
    (53, "The Lord of the Rings",              "J.R.R. Tolkien"),
    (54, "Loving",                             "Henry Green"),
    (55, "Lucky Jim",                          "Kingsley Amis"),
    (56, "The Man Who Loved Children",         "Christina Stead"),
    (57, "Midnight's Children",                "Salman Rushdie"),
    (58, "Money",                              "Martin Amis"),
    (59, "The Moviegoer",                      "Walker Percy"),
    (60, "Mrs. Bridge",                        "Evan S. Connell"),
    (61, "Naked Lunch",                        "William S. Burroughs"),
    (62, "Native Son",                         "Richard Wright"),
    (63, "Neuromancer",                        "William Gibson"),
    (64, "Never Let Me Go",                    "Kazuo Ishiguro"),
    (65, "1984",                               "George Orwell"),
    (66, "On the Road",                        "Jack Kerouac"),
    (67, "One Flew Over the Cuckoo's Nest",    "Ken Kesey"),
    (68, "The Painted Bird",                   "Jerzy Kosinski"),
    (69, "Pale Fire",                          "Vladimir Nabokov"),
    (70, "A Passage to India",                 "E.M. Forster"),
    (71, "Play It as It Lays",                 "Joan Didion"),
    (72, "Portnoy's Complaint",                "Philip Roth"),
    (73, "Possession",                         "A.S. Byatt"),
    (74, "The Power and the Glory",            "Graham Greene"),
    (75, "The Prime of Miss Jean Brodie",      "Muriel Spark"),
    (76, "Rabbit, Run",                        "John Updike"),
    (77, "Ragtime",                            "E.L. Doctorow"),
    (78, "The Recognitions",                   "William Gaddis"),
    (79, "Red Harvest",                        "Dashiell Hammett"),
    (80, "Revolutionary Road",                 "Richard Yates"),
    (81, "The Sheltering Sky",                 "Paul Bowles"),
    (82, "Slaughterhouse-Five",                "Kurt Vonnegut"),
    (83, "Snow Crash",                         "Neal Stephenson"),
    (84, "The Sot-Weed Factor",                "John Barth"),
    (85, "The Sound and the Fury",             "William Faulkner"),
    (86, "The Sportswriter",                   "Richard Ford"),
    (87, "The Spy Who Came in from the Cold",  "John le Carre"),
    (88, "The Sun Also Rises",                 "Ernest Hemingway"),
    (89, "Their Eyes Were Watching God",       "Zora Neale Hurston"),
    (90, "Things Fall Apart",                  "Chinua Achebe"),
    (91, "To Kill a Mockingbird",              "Harper Lee"),
    (92, "To the Lighthouse",                  "Virginia Woolf"),
    (93, "Tropic of Cancer",                   "Henry Miller"),
    (94, "Ubik",                               "Philip K. Dick"),
    (95, "Under the Net",                      "Iris Murdoch"),
    (96, "Under the Volcano",                  "Malcolm Lowry"),
    (97, "Watchmen",                           "Alan Moore"),
    (98, "White Noise",                        "Don DeLillo"),
    (99, "White Teeth",                        "Zadie Smith"),
    (100,"Wide Sargasso Sea",                  "Jean Rhys"),
]

time_df = pd.DataFrame(TIME_100_NOVELS, columns=['time_rank', 'title', 'author'])
time_df['time_100'] = True
time_df['medium'] = 'novel'
time_df['norm_title'] = time_df['title'].apply(normalize)
print(f"TIME's 100: {len(time_df)} novels")

NOBEL_CACHE = os.path.join(CACHE_DIR, 'sparql_nobel_novels.csv')
if os.path.exists(NOBEL_CACHE):
    nobel_raw = pd.read_csv(NOBEL_CACHE)
else:
    query = """
SELECT DISTINCT ?work ?workLabel ?authorLabel ?pubdate WHERE {
  ?author wdt:P166 wd:Q37922 .
  ?work wdt:P50 ?author .
  ?work wdt:P31 ?type .
  VALUES ?type { wd:Q7725634 wd:Q571 wd:Q8261 }
  ?work wikibase:sitelinks ?sitelinks .
  FILTER(?sitelinks >= 20)
  OPTIONAL { ?work wdt:P577 ?pubdate . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" . }
}
"""
    nobel_raw = run_sparql(query)
    nobel_raw.to_csv(NOBEL_CACHE, index=False)

if len(nobel_raw) > 0:
    nobel_df = nobel_raw.rename(columns={'workLabel': 'title', 'authorLabel': 'author'}).copy()
    nobel_df['year'] = nobel_df['pubdate'].str.extract(r'(\d{4})').astype(float) \
                       if 'pubdate' in nobel_df.columns else np.nan
    nobel_df['time_100'] = False
    nobel_df['nobel_author'] = True
    nobel_df['medium'] = 'novel'
    nobel_df['norm_title'] = nobel_df['title'].apply(normalize)
    nobel_df = nobel_df[nobel_df['title'].str.len() > 1].drop_duplicates('norm_title').reset_index(drop=True)
    time_df = pd.concat([time_df, nobel_df[~nobel_df['norm_title'].isin(time_df['norm_title'])]],
                        ignore_index=True)
    print(f"TIME's 100 + Nobel: {len(time_df)} total")


TIME's 100: 100 novels
TIME's 100 + Nobel: 178 total


In [13]:
pulitzer_df = pd.DataFrame()
for page_name in ['Pulitzer Prize for Fiction', 'Pulitzer Prize for the Novel',
                   'Pulitzer Prize for the Novel or Play']:
    tables_pulitzer = get_tables(page_name)
    if tables_pulitzer:
        pulitzer_df = parse_novel_table(tables_pulitzer, min_rows=20)
        if len(pulitzer_df) > 0:
            print(f'Pulitzer from: "{page_name}"')
            break

if len(pulitzer_df) == 0:
    PULITZER_CACHE = os.path.join(CACHE_DIR, 'sparql_pulitzer.csv')
    if os.path.exists(PULITZER_CACHE):
        pulitzer_raw = pd.read_csv(PULITZER_CACHE)
    else:
        query = """
SELECT DISTINCT ?work ?workLabel ?authorLabel ?pubdate WHERE {
  ?work wdt:P166 wd:Q131135 .
  OPTIONAL { ?work wdt:P50 ?author . }
  OPTIONAL { ?work wdt:P577 ?pubdate . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" . }
}
"""
        pulitzer_raw = run_sparql(query)
        pulitzer_raw.to_csv(PULITZER_CACHE, index=False)
    if len(pulitzer_raw) > 0:
        pulitzer_df = pulitzer_raw.rename(
            columns={'workLabel': 'title', 'authorLabel': 'author'}
        ).copy()
        pulitzer_df['pulitzer_year'] = pulitzer_df['pubdate'].str.extract(r'(\d{4})').astype(float) \
                                       if 'pubdate' in pulitzer_df.columns else np.nan

if len(pulitzer_df) == 0:
    print('Warning: no Pulitzer data found.')
    pulitzer_df = pd.DataFrame(columns=['title', 'author'])

pulitzer_df['medium'] = 'novel'
pulitzer_df['pulitzer'] = True
pulitzer_df['norm_title'] = pulitzer_df['title'].apply(normalize) if 'title' in pulitzer_df.columns else ''
if 'year' in pulitzer_df.columns and 'pulitzer_year' not in pulitzer_df.columns:
    pulitzer_df = pulitzer_df.rename(columns={'year': 'pulitzer_year'})
pulitzer_df = pulitzer_df[pulitzer_df['title'].str.len() > 1].reset_index(drop=True)
print(f'Pulitzer: {len(pulitzer_df)} novels')


Pulitzer from: "Pulitzer Prize for Fiction"


Pulitzer: 62 novels


In [14]:
tables_booker = get_tables('Booker Prize')
booker_df = pd.DataFrame()
for t in tables_booker:
    t.columns = [str(c).strip() for c in t.columns]
    cols_lower = {c.lower(): c for c in t.columns}
    if 'title' in cols_lower and len(t) >= 20:
        df = pd.DataFrame()
        df['title']      = t[cols_lower['title']].astype(str).str.strip()
        df['author']     = t[cols_lower['author']].astype(str).str.strip() if 'author' in cols_lower else ''
        df['booker_year']= pd.to_numeric(t[cols_lower['year']], errors='coerce') if 'year' in cols_lower else np.nan
        df = df[df['title'].str.len() > 1].dropna(subset=['title'])
        df = df[~df['title'].str.match(r'^[\d\W]+$')]
        booker_df = df.reset_index(drop=True)
        break

if len(booker_df) == 0:
    print('Warning: no Booker data found.')
    booker_df = pd.DataFrame(columns=['title', 'author', 'booker_year'])

booker_df['medium'] = 'novel'
booker_df['booker'] = True
booker_df['norm_title'] = booker_df['title'].apply(normalize)
print(f'Booker Prize: {len(booker_df)} novels')


Booker Prize: 60 novels


In [15]:
tables_nba = get_tables('National Book Award for Fiction')
nba_rows = []
for t in tables_nba:
    t.columns = [str(c).strip() for c in t.columns]
    cols_lower = {c.lower(): c for c in t.columns}
    if 'title' not in cols_lower:
        continue
    result_col = cols_lower.get('result', None)
    winners = t[t[result_col].astype(str).str.lower().str.contains('winner|won', na=False)] \
              if result_col else t
    for _, row in winners.iterrows():
        title  = str(row[cols_lower['title']]).strip()
        author = str(row[cols_lower['author']]).strip() if 'author' in cols_lower else ''
        year   = row[cols_lower['year']] if 'year' in cols_lower else np.nan
        if title and len(title) > 1 and not re.match(r'^[\d\W]+$', title):
            nba_rows.append({'title': title, 'author': author, 'nba_year': year})

if nba_rows:
    nba_df = pd.DataFrame(nba_rows).drop_duplicates(subset=['title']).reset_index(drop=True)
else:
    print('Warning: no NBA data found.')
    nba_df = pd.DataFrame(columns=['title', 'author', 'nba_year'])

nba_df['medium'] = 'novel'
nba_df['national_book_award'] = True
nba_df['norm_title'] = nba_df['title'].apply(normalize)
print(f'National Book Award: {len(nba_df)} novels')


National Book Award: 83 novels

---
## Section 3: Combine & Deduplicate Lists

Merge all film sources and all novel sources on normalized title.
Track which lists each work appears on and at what rank.

In [16]:
def combine_lists(frames, title_col='title', norm_col='norm_title'):
    """Merge multiple list DataFrames on normalized title, avoiding column conflicts."""
    if not frames:
        return pd.DataFrame()

    all_norms = pd.concat(
        [f[[title_col, norm_col, 'medium']].copy() for f in frames if len(f) > 0],
        ignore_index=True
    ).drop_duplicates(subset=norm_col)
    merged = all_norms.copy()

    year_dfs      = []
    author_map    = {}
    director_map  = {}

    for f in frames:
        if len(f) == 0:
            continue
        # Collect year separately to avoid merge suffix conflicts
        if 'year' in f.columns:
            ydf = f[[norm_col, 'year']].dropna(subset=['year']).drop_duplicates(norm_col)
            year_dfs.append(ydf)
        # Collect first non-empty author / director
        for attr, amap in [('author', author_map), ('director', director_map)]:
            if attr in f.columns:
                for _, row in f[[norm_col, attr]].dropna(subset=[attr]).iterrows():
                    if row[norm_col] not in amap and str(row[attr]).strip():
                        amap[row[norm_col]] = str(row[attr]).strip()
        # Merge all other columns (flag/rank cols, _year suffix cols)
        skip = {title_col, norm_col, 'medium', 'director', 'author', 'year'}
        extra_cols = [c for c in f.columns if c not in skip]
        if extra_cols:
            sub = f[[norm_col] + extra_cols].drop_duplicates(subset=norm_col)
            merged = merged.merge(sub, on=norm_col, how='left')

    # Coalesce year: minimum across sources
    if year_dfs:
        all_years = pd.concat(year_dfs).groupby(norm_col)['year'].min().reset_index()
        merged = merged.merge(all_years, on=norm_col, how='left')
    # Incorporate _year suffix cols (booker_year, pulitzer_year, etc.)
    year_sfx = [c for c in merged.columns if c.endswith('_year')]
    if year_sfx:
        if 'year' not in merged.columns:
            merged['year'] = np.nan
        for c in year_sfx:
            merged['year'] = merged['year'].fillna(merged[c])

    merged['author']   = merged[norm_col].map(author_map).fillna('')
    merged['director'] = merged[norm_col].map(director_map).fillna('')
    return merged


# Films
film_sources = []
for df, name in [(afi_df, 'AFI'), (ss_df, 'S&S'), (imdb_df, 'Awards'), (ebert_df, 'Criterion')]:
    if len(df) > 0:
        film_sources.append(df)
        print(f'  {name}: {len(df)} films')

films = combine_lists(film_sources)
rank_flag_cols = ['afi_rank', 'ss_rank', 'imdb_rank', 'ebert_great', 'criterion']
present_cols = [c for c in rank_flag_cols if c in films.columns]
films['list_count'] = films[present_cols].notna().sum(axis=1)
print(f'\nFilms corpus: {len(films):,}')
print(f'  On 2+ lists: {(films["list_count"] >= 2).sum()}')


  AFI: 100 films
  S&S: 124 films
  Awards: 157 films
  Criterion: 84 films



Films corpus: 391
  On 2+ lists: 145


In [17]:
# ── Novels ────────────────────────────────────────────────────────────────────
novel_sources = []
for df, name in [(ml_df, 'ModernLib'), (time_df, 'TIME'), (pulitzer_df, 'Pulitzer'),
                  (booker_df, 'Booker'), (nba_df, 'NBA')]:
    if len(df) > 0:
        novel_sources.append(df)
        print(f'  {name}: {len(df)} novels')

novels = combine_lists(novel_sources)

novel_flag_cols = ['ml_rank', 'time_100', 'pulitzer', 'booker', 'national_book_award']
present_novel_cols = [c for c in novel_flag_cols if c in novels.columns]
novels['list_count'] = novels[present_novel_cols].notna().sum(axis=1)

print(f'\nNovels corpus (deduplicated): {len(novels):,}')
print(f'  On 2+ lists: {(novels["list_count"] >= 2).sum()}')
print(f'  On 1  list:  {(novels["list_count"] == 1).sum()}')

  ModernLib: 100 novels
  TIME: 178 novels
  Pulitzer: 62 novels
  Booker: 60 novels
  NBA: 83 novels



Novels corpus (deduplicated): 417
  On 2+ lists: 62
  On 1  list:  355


In [18]:
# ── Combine films + novels into one corpus ────────────────────────────────────
corpus = pd.concat([films, novels], ignore_index=True)
corpus['norm_title'] = corpus['title'].apply(normalize)

# Deduplicate (same title appearing in both film and novel lists is kept as separate rows)
corpus = corpus.drop_duplicates(subset=['norm_title', 'medium']).reset_index(drop=True)

# Assign internal work ID
corpus['work_id'] = ['W{:05d}'.format(i) for i in range(len(corpus))]

print(f'Combined corpus: {len(corpus):,} works')
print(f'  Films:  {(corpus["medium"] == "film").sum():,}')
print(f'  Novels: {(corpus["medium"] == "novel").sum():,}')

Combined corpus: 808 works
  Films:  391
  Novels: 417


---
## Section 4: Wikidata Entity Linking

Resolve each work to a Wikidata QID using the `wbsearchentities` API.
This enables metadata enrichment, cluster building via P144, and joining to the concept dataset.

Results are cached in `data/cache/nb09/entity_linking.json`.

In [19]:
ENTITY_LINK_CACHE = os.path.join(CACHE_DIR, 'entity_linking.json')

# Load existing cache
if os.path.exists(ENTITY_LINK_CACHE):
    with open(ENTITY_LINK_CACHE, 'r') as f:
        link_cache = json.load(f)
    print(f'Loaded entity link cache: {len(link_cache):,} entries')
else:
    link_cache = {}
    print('Starting fresh entity link cache.')


def search_wikidata_qid(title, medium, year=None):
    """
    Search Wikidata for a work by title and medium.
    Returns (qid, label, description) or (None, None, None).
    """
    cache_key = f'{normalize(title)}|{medium}'
    if cache_key in link_cache:
        return link_cache[cache_key]

    medium_keywords = {
        'film':  ['film', 'movie', 'documentary', 'animated'],
        'novel': ['novel', 'book', 'fiction', 'literature'],
    }
    keywords = medium_keywords.get(medium, [])

    try:
        resp = SESSION.get(
            WIKIDATA_API,
            params={'action': 'wbsearchentities', 'search': title,
                    'language': 'en', 'type': 'item', 'limit': 10, 'format': 'json'},
            timeout=15,
        )
        resp.raise_for_status()
        results = resp.json().get('search', [])

        if not results:
            link_cache[cache_key] = (None, None, None)
            return (None, None, None)

        # Prefer results whose description matches the medium
        for r in results:
            desc = r.get('description', '').lower()
            if any(k in desc for k in keywords):
                # Optionally verify year if provided
                if year and not pd.isna(year):
                    if str(int(year)) in desc or str(int(year)-1) in desc:
                        result = (r['id'], r.get('label', ''), r.get('description', ''))
                        link_cache[cache_key] = result
                        return result
                result = (r['id'], r.get('label', ''), r.get('description', ''))
                link_cache[cache_key] = result
                return result

        # Fall back to first result
        r = results[0]
        result = (r['id'], r.get('label', ''), r.get('description', ''))
        link_cache[cache_key] = result
        return result

    except Exception as e:
        link_cache[cache_key] = (None, None, None)
        return (None, None, None)


print(f'Entity linker ready. {len(corpus):,} works to resolve.')

Starting fresh entity link cache.
Entity linker ready. 808 works to resolve.


In [20]:
# Run entity linking — hits Wikidata API for each uncached work
# Saves cache every 100 items to avoid losing progress

qids, wd_labels, wd_descs = [], [], []

for i, row in corpus.iterrows():
    if i > 0 and i % 100 == 0:
        print(f'  {i}/{len(corpus)} resolved...', end='\r')
        with open(ENTITY_LINK_CACHE, 'w') as f:
            json.dump(link_cache, f)

    qid, label, desc = search_wikidata_qid(
        row['title'], row['medium'],
        year=row.get('year', np.nan)
    )
    qids.append(qid)
    wd_labels.append(label)
    wd_descs.append(desc)
    time.sleep(0.1)  # polite rate limiting

# Final cache save
with open(ENTITY_LINK_CACHE, 'w') as f:
    json.dump(link_cache, f)

corpus['qid']         = qids
corpus['wd_label']    = wd_labels
corpus['wd_desc']     = wd_descs

linked = corpus['qid'].notna().sum()
print(f'\nEntity linking complete.')
print(f'  Linked: {linked:,} / {len(corpus):,} ({linked/len(corpus)*100:.1f}%)')
print(f'  Unlinked: {corpus["qid"].isna().sum():,}')

# Review unlinked works — may need manual QID
print('\nSample unlinked works:')
print(corpus[corpus['qid'].isna()][['title', 'medium', 'year']].head(20).to_string())


Entity linking complete.
  Linked: 272 / 808 (33.7%)
  Unlinked: 536

Sample unlinked works:
                                             title medium    year
101                                     The Matrix   film  1999.0
102                                      Gladiator   film  2000.0
103                                The Dark Knight   film  2008.0
105                                     Fight Club   film  1999.0
106                                  Jurassic Park   film  1993.0
107                                 The Terminator   film  1984.0
109                                 V for Vendetta   film  2005.0
110       Harry Potter and the Philosopher's Stone   film  2001.0
111                                   Interstellar   film  2014.0
113                                   The Avengers   film  2012.0
114        Harry Potter and the Chamber of Secrets   film  2002.0
115                                     Braveheart   film  1995.0
117  The Lord of the Rings: The Return of the Ki

---
## Section 5: Wikidata Metadata Enrichment

For all QID-linked works, fetch via SPARQL:
- `P144` — based on (links film adaptations back to source novel)
- `P57` / `P50` — director / author
- `P136` — genre
- `P577` — publication/release date
- `P2047` — duration in minutes (films)
- `P1657` — MPAA rating
- `P407` — language
- Sitelinks — Wikipedia cross-language presence (importance signal)

In [21]:
METADATA_CACHE = os.path.join(CACHE_DIR, 'wikidata_metadata.csv')

def fetch_metadata_for_qids(qids_list):
    """Fetch Wikidata metadata for a list of QIDs via SPARQL in batches of 150."""
    all_results = []
    batches = [qids_list[i:i+150] for i in range(0, len(qids_list), 150)]

    for b_idx, batch in enumerate(batches):
        values = ' '.join(f'wd:{q}' for q in batch if q and str(q).startswith('Q'))
        if not values:
            continue

        query = f"""
SELECT DISTINCT ?item ?sitelinks ?based_on ?genreLabel ?runtime ?mpaaLabel
                ?pubdate ?directorLabel ?authorLabel WHERE {{
  VALUES ?item {{ {values} }}
  OPTIONAL {{ ?item wikibase:sitelinks ?sitelinks . }}
  OPTIONAL {{ ?item wdt:P144 ?based_on . }}
  OPTIONAL {{ ?item wdt:P136 ?genre . }}
  OPTIONAL {{ ?item wdt:P2047 ?runtime . }}
  OPTIONAL {{
    ?item wdt:P1657 ?mpaaItem .
    ?mpaaItem rdfs:label ?mpaaLabel .
    FILTER(LANG(?mpaaLabel) = "en")
  }}
  OPTIONAL {{ ?item wdt:P577 ?pubdate . }}
  OPTIONAL {{ ?item wdt:P57 ?director . }}
  OPTIONAL {{ ?item wdt:P50 ?author . }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
}}
"""
        print(f'  Metadata batch {b_idx+1}/{len(batches)} ({len(batch)} QIDs)...', end=' ')
        result = run_sparql(query)
        print(f'{len(result):,} rows')
        if len(result) > 0:
            all_results.append(result)
        time.sleep(5)

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    return pd.DataFrame()


if os.path.exists(METADATA_CACHE):
    metadata_raw = pd.read_csv(METADATA_CACHE)
    print(f'Loaded metadata cache: {len(metadata_raw):,} rows')
else:
    qids_to_fetch = corpus['qid'].dropna().unique().tolist()
    print(f'Fetching metadata for {len(qids_to_fetch):,} QIDs...')
    metadata_raw = fetch_metadata_for_qids(qids_to_fetch)
    metadata_raw.to_csv(METADATA_CACHE, index=False)
    print(f'Saved metadata cache: {len(metadata_raw):,} rows')

print(f'\nMetadata columns: {list(metadata_raw.columns)}')

Fetching metadata for 271 QIDs...
  Metadata batch 1/2 (150 QIDs)... 

4,069 rows


  Metadata batch 2/2 (121 QIDs)... 

454 rows


Saved metadata cache: 4,523 rows

Metadata columns: ['item', 'mpaaLabel', 'sitelinks', 'runtime', 'pubdate', 'genreLabel', 'directorLabel', 'based_on', 'authorLabel']


In [22]:
# Process and collapse metadata — one row per QID
meta = metadata_raw.copy()
meta['qid'] = meta['item'].str.extract(r'(Q\d+)$')
meta['based_on_qid'] = meta['based_on'].str.extract(r'(Q\d+)$')

# Convert numeric fields
meta['sitelinks'] = pd.to_numeric(meta.get('sitelinks', np.nan), errors='coerce')
meta['runtime']   = pd.to_numeric(meta.get('runtime', np.nan), errors='coerce')

# Extract year from pubdate
if 'pubdate' in meta.columns:
    meta['pub_year'] = meta['pubdate'].str.extract(r'(\d{4})').astype(float)

# Collapse to one row per QID: max sitelinks, first based_on, aggregate genres/ratings
agg = {
    'sitelinks':      ('sitelinks', 'max'),
    'based_on_qid':   ('based_on_qid', 'first'),
    'genre':          ('genreLabel', lambda x: '; '.join(sorted(set(x.dropna()))) if x.notna().any() else ''),
    'runtime_min':    ('runtime', 'max'),
    'mpaa_rating':    ('mpaaLabel', 'first') if 'mpaaLabel' in meta.columns else ('qid', 'first'),
    'director':       ('directorLabel', 'first') if 'directorLabel' in meta.columns else ('qid', 'first'),
    'author':         ('authorLabel', 'first') if 'authorLabel' in meta.columns else ('qid', 'first'),
}
# Only include columns that exist
agg = {k: v for k, v in agg.items() if v[0] in meta.columns}
if 'pub_year' in meta.columns:
    agg['pub_year'] = ('pub_year', 'min')

meta_collapsed = meta.groupby('qid').agg(**agg).reset_index()

print(f'Collapsed metadata: {len(meta_collapsed):,} QIDs')
print(f'  With based_on_qid: {meta_collapsed["based_on_qid"].notna().sum():,}')
print(f'  With runtime:      {meta_collapsed["runtime_min"].notna().sum():,}')
print(f'  With sitelinks:    {meta_collapsed["sitelinks"].notna().sum():,}')
print(meta_collapsed.head(5).to_string())

Collapsed metadata: 271 QIDs
  With based_on_qid: 76
  With runtime:      168
  With sitelinks:    271
        qid  sitelinks based_on_qid                                                                                                                                                                                                                                  genre  runtime_min mpaa_rating           director        author  pub_year
0   Q102225         89       Q46751                                                                                                                                                                   adventure film; children's film; fantasy film; film based on a novel        151.0       PG-13        Mike Newell           NaN    2005.0
1  Q1029093          5          NaN                                                                                                                                                                                                

In [23]:
# Merge metadata into corpus
corpus = corpus.merge(meta_collapsed, on='qid', how='left')

# Coalesce year: prefer pub_year from Wikidata over scraped year
if 'pub_year' in corpus.columns:
    corpus['year'] = corpus['pub_year'].fillna(corpus.get('year', np.nan))

# Coalesce director/author: prefer Wikidata over scraped
for attr in ['director', 'author']:
    wikidata_col = attr
    scraped_col  = attr + '_x' if attr + '_x' in corpus.columns else None
    if scraped_col and wikidata_col in corpus.columns:
        corpus[attr] = corpus[wikidata_col].fillna(corpus[scraped_col])
    elif wikidata_col in corpus.columns:
        corpus[attr] = corpus[wikidata_col]

# Rename suffixed columns from merge
corpus.columns = [re.sub(r'_[xy]$', '', c) if c.endswith(('_x', '_y')) else c
                  for c in corpus.columns]
corpus = corpus.loc[:, ~corpus.columns.duplicated(keep='first')]

print(f'Corpus after metadata merge: {len(corpus):,} rows')
print(f'Columns: {list(corpus.columns)}')

Corpus after metadata merge: 808 rows
Columns: ['title', 'norm_title', 'medium', 'afi_rank', 'sitelinks', 'ss_rank', 'awards', 'imdb_rank', 'criterion_spine', 'ebert_great', 'criterion', 'year', 'author', 'director', 'list_count', 'ml_rank', 'time_rank', 'time_100', 'work', 'pubdate', 'nobel_author', 'pulitzer_year', 'rank', 'pulitzer', 'booker_year', 'booker', 'nba_year', 'national_book_award', 'work_id', 'qid', 'wd_label', 'wd_desc', 'based_on_qid', 'genre', 'runtime_min', 'mpaa_rating', 'pub_year']


---
## Section 6: Work Importance Composite Score

Compute a composite importance score from:
1. **Canonical list ranks** — percentile rank on AFI, Sight & Sound, IMDb (ranked lists)
2. **List membership** — presence on Criterion, Ebert, Pulitzer, Booker, NBA (membership lists)
3. **Wikidata sitelinks** — cross-language Wikipedia presence
4. **Composite dataset score** — works already in `cultural_salience_composite_v2.csv`

Each signal is percentile-ranked 0–100, then averaged (consistent with prior notebooks).

In [24]:
def pctile(series):
    """Percentile rank, 0–100, higher = more important. NaN stays NaN."""
    return series.rank(pct=True, na_option='keep') * 100


# ── Films: rank signals (lower rank number = more important → invert) ──────────
for rank_col, pctile_col in [('afi_rank', 'pctile_afi'),
                               ('ss_rank',  'pctile_ss'),
                               ('imdb_rank','pctile_imdb')]:
    if rank_col in corpus.columns:
        # Invert: rank 1 should get highest percentile
        inverted = corpus[rank_col].max() - corpus[rank_col]
        corpus[pctile_col] = pctile(inverted)

# ── Membership signals (binary → treat as 100 if present, else NaN) ───────────
for flag_col, pctile_col in [('ebert_great',         'pctile_ebert'),
                               ('criterion',           'pctile_criterion'),
                               ('time_100',            'pctile_time'),
                               ('pulitzer',            'pctile_pulitzer'),
                               ('booker',              'pctile_booker'),
                               ('national_book_award', 'pctile_nba')]:
    if flag_col in corpus.columns:
        corpus[pctile_col] = np.where(corpus[flag_col].fillna(False).astype(bool), 100, np.nan)

# ── Sitelinks signal ──────────────────────────────────────────────────────────
if 'sitelinks' in corpus.columns:
    corpus['pctile_sitelinks'] = pctile(corpus['sitelinks'])

# ── Concept dataset score (for works already in composite v2) ─────────────────
if 'composite_score' in composite.columns:
    comp_subset = composite[['norm', 'composite_score']].rename(
        columns={'norm': 'norm_title', 'composite_score': 'concept_dataset_score'}
    ).drop_duplicates('norm_title')
    corpus = corpus.merge(comp_subset, on='norm_title', how='left')
    corpus['pctile_concept_dataset'] = pctile(corpus['concept_dataset_score'])

# ── Compute composite ─────────────────────────────────────────────────────────
importance_signal_cols = [c for c in corpus.columns if c.startswith('pctile_')]
corpus['n_importance_signals'] = corpus[importance_signal_cols].notna().sum(axis=1)
corpus['importance_score'] = corpus[importance_signal_cols].mean(axis=1)

print('Importance signals:', importance_signal_cols)
print(f'\nImportance score distribution:')
print(corpus['importance_score'].describe().round(1))
print(f'\nTop 20 films by importance:')
top_films = corpus[corpus['medium']=='film'].nlargest(20, 'importance_score')
print(top_films[['title','year','importance_score','n_importance_signals','list_count']].to_string())
print(f'\nTop 20 novels by importance:')
top_novels = corpus[corpus['medium']=='novel'].nlargest(20, 'importance_score')
print(top_novels[['title','year','importance_score','n_importance_signals','list_count']].to_string())

Importance signals:

 ['pctile_afi', 'pctile_ss', 'pctile_imdb', 'pctile_ebert', 'pctile_criterion', 'pctile_time', 'pctile_pulitzer', 'pctile_booker', 'pctile_nba', 'pctile_sitelinks', 'pctile_concept_dataset']

Importance score distribution:
count    765.0
mean      69.6
std       27.2
min        0.8
25%       50.9
50%       71.5
75%      100.0
max      100.0
Name: importance_score, dtype: float64

Top 20 films by importance:
                          title    year  importance_score  n_importance_signals  list_count
311                       Ikiru  1952.0             100.0                     2           2
313                     Yojimbo  1961.0             100.0                     2           2
315                 Tokyo Story  1953.0             100.0                     2           2
316                 Late Spring  1949.0             100.0                     2           2
324                    Amarcord  1973.0             100.0                     2           2
325             Bicycle Thieves  1948

---
## Section 7: Adaptation Clusters

Use Wikidata `P144` (based on) to link film adaptations to their source novels.
Works sharing a common origin are grouped into a cluster.

**Cluster examples:**
- Cluster: *Jurassic Park* → novel (Crichton, 1990) + film (1993)
- Cluster: *To Kill a Mockingbird* → novel (Lee, 1960) + film (1962)
- Cluster: *Les Misérables* → novel (Hugo, 1862) + multiple films

The `recommended_entry` field is left for human review in Notebook 12.

In [25]:
# Build cluster assignments
# Strategy: follow P144 (based_on_qid) chains — a film points to its source novel.
# The source novel's QID becomes the cluster root.

qid_to_work_id = {}
for _, row in corpus.iterrows():
    if pd.notna(row.get('qid')):
        qid_to_work_id[row['qid']] = row['work_id']

def find_cluster_root(qid, based_on_map, visited=None):
    """Follow P144 chain to find the root source work."""
    if visited is None:
        visited = set()
    if qid in visited or qid not in based_on_map:
        return qid
    visited.add(qid)
    return find_cluster_root(based_on_map[qid], based_on_map, visited)

# Build based_on map: child_qid -> parent_qid
based_on_map = {}
for _, row in corpus.iterrows():
    if pd.notna(row.get('qid')) and pd.notna(row.get('based_on_qid')):
        # Only link if parent QID is also in our corpus
        if row['based_on_qid'] in qid_to_work_id:
            based_on_map[row['qid']] = row['based_on_qid']

print(f'Works with P144 based_on links to corpus: {len(based_on_map):,}')

# Assign cluster_root_qid
def get_cluster_root(row):
    if pd.isna(row.get('qid')):
        return row['work_id']  # self-cluster
    return find_cluster_root(row['qid'], based_on_map)

corpus['cluster_root_qid'] = corpus.apply(get_cluster_root, axis=1)

# Assign numeric cluster IDs
unique_roots = corpus['cluster_root_qid'].unique()
root_to_cluster_id = {r: f'C{i:04d}' for i, r in enumerate(unique_roots)}
corpus['cluster_id'] = corpus['cluster_root_qid'].map(root_to_cluster_id)

# Find multi-work clusters (actual novel+film pairs)
cluster_sizes = corpus['cluster_id'].value_counts()
multi_clusters = cluster_sizes[cluster_sizes > 1].index
print(f'\nClusters with multiple works: {len(multi_clusters):,}')

print('\nSample multi-work clusters:')
sample = corpus[corpus['cluster_id'].isin(multi_clusters[:10])]
for cid, group in sample.groupby('cluster_id'):
    print(f'  [{cid}]')
    for _, r in group.iterrows():
        print(f'    {r["medium"]:6s} | {r["title"]} ({int(r["year"]) if pd.notna(r["year"]) else "?"}) | importance={r["importance_score"]:.0f}')

Works with P144 based_on links to corpus: 4



Clusters with multiple works: 5

Sample multi-work clusters:
  [C0005]
    film   | Gone with the Wind (1939) | importance=75
    novel  | Gone with the Wind (1936) | importance=70
  [C0012]
    film   | Star Wars (1977) | importance=87
    film   | Star Wars: Episode IV – A New Hope (1977) | importance=83
  [C0024]
    film   | To Kill a Mockingbird (1962) | importance=56
    novel  | To Kill a Mockingbird (1960) | importance=68
  [C0029]
    film   | Apocalypse Now (1979) | importance=54
    novel  | Heart of Darkness (1899) | importance=71
  [C0090]
    film   | Sophie's Choice (1982) | importance=36
    novel  | Sophie's Choice (1979) | importance=81


In [26]:
# Build work_clusters.csv — one row per cluster
cluster_rows = []

for cid, group in corpus.groupby('cluster_id'):
    novels = group[group['medium'] == 'novel']
    films  = group[group['medium'] == 'film']

    # Canonical title: prefer the novel title (it's typically the source),
    # fall back to the highest-importance work title
    if len(novels) > 0:
        canonical_title = novels.iloc[0]['title']
    else:
        canonical_title = group.nlargest(1, 'importance_score').iloc[0]['title']

    cluster_rows.append({
        'cluster_id':       cid,
        'canonical_title':  canonical_title,
        'n_works':          len(group),
        'has_novel':        len(novels) > 0,
        'has_film':         len(films) > 0,
        'novel_work_ids':   '; '.join(novels['work_id'].tolist()),
        'film_work_ids':    '; '.join(films['work_id'].tolist()),
        'novel_qids':       '; '.join(novels['qid'].dropna().tolist()),
        'film_qids':        '; '.join(films['qid'].dropna().tolist()),
        'recommended_entry_work_id': '',  # to be filled in Notebook 12
        'recommendation_notes':      '',  # to be filled in Notebook 12
    })

clusters_df = pd.DataFrame(cluster_rows)
adaptation_clusters = clusters_df[clusters_df['has_novel'] & clusters_df['has_film']]

print(f'Total clusters: {len(clusters_df):,}')
print(f'  Adaptation clusters (novel + film): {len(adaptation_clusters):,}')
print(f'  Novel-only clusters: {((clusters_df["has_novel"]) & (~clusters_df["has_film"])).sum():,}')
print(f'  Film-only clusters:  {((~clusters_df["has_novel"]) & (clusters_df["has_film"])).sum():,}')
print(f'\nAdaptation clusters sample:')
print(adaptation_clusters[['cluster_id','canonical_title','novel_work_ids','film_work_ids']].head(20).to_string())

Total clusters: 803
  Adaptation clusters (novel + film): 4
  Novel-only clusters: 413
  Film-only clusters:  386

Adaptation clusters sample:
   cluster_id        canonical_title novel_work_ids film_work_ids
5       C0005     Gone with the Wind         W00510        W00005
24      C0024  To Kill a Mockingbird         W00539        W00024
29      C0029      Heart of Darkness         W00457        W00029
90      C0090        Sophie's Choice         W00486        W00090


---
## Section 8: Age Tier Assignment

Four tiers: `child` (≤10), `middle` (11–13), `ya` (14–17), `adult`

**Films:** MPAA rating → tier (G/PG → child/middle, PG-13 → ya, R/NR → adult)  
**Novels:** Wikidata genre (P136) for YA/children's flags; page count heuristics; default adult

In [27]:
FILM_MPAA_TO_TIER = {
    'G':     'child',
    'PG':    'middle',
    'PG-13': 'ya',
    'R':     'adult',
    'NC-17': 'adult',
    'NR':    'adult',
    'Unrated': 'adult',
}

# Genres that signal YA or children's literature
CHILD_GENRE_KEYWORDS = ['children', "children's", 'juvenile', 'picture book']
YA_GENRE_KEYWORDS    = ['young adult', 'ya fiction', 'teen']


def assign_age_tier(row):
    """Return age tier string based on MPAA rating (films) or genre flags (novels)."""
    medium = row.get('medium', '')
    genre  = str(row.get('genre', '')).lower()

    if medium == 'film':
        mpaa = str(row.get('mpaa_rating', '')).strip()
        tier = FILM_MPAA_TO_TIER.get(mpaa, None)
        if tier:
            return tier
        # Older films predate MPAA — check genre/description for content signals
        desc = str(row.get('wd_desc', '')).lower()
        if any(k in genre or k in desc for k in CHILD_GENRE_KEYWORDS):
            return 'child'
        return 'adult'  # default for unrated films

    elif medium == 'novel':
        if any(k in genre for k in CHILD_GENRE_KEYWORDS):
            return 'child'
        if any(k in genre for k in YA_GENRE_KEYWORDS):
            return 'ya'
        return 'adult'  # default — most canonical novels are adult

    return 'adult'


corpus['age_tier'] = corpus.apply(assign_age_tier, axis=1)

print('Age tier distribution:')
print(corpus.groupby(['medium', 'age_tier']).size().to_string())
print(f'\nNote: most novels default to adult; YA/children flagged via Wikidata genre.')
print(f'Manual review of age tier recommended for well-known YA/children works.')

Age tier distribution:
medium  age_tier
film    adult       338
        child        13
        middle       26
        ya           14
novel   adult       416
        ya            1

Note: most novels default to adult; YA/children flagged via Wikidata genre.
Manual review of age tier recommended for well-known YA/children works.


---
## Section 9: Wikidata P921 Expansion

Query Wikidata for films and novels where the **main subject** (P921) maps to a
high-salience concept in our dataset. This surfaces concept-vehicle works that
don't appear on canonical ranking lists — e.g. *Amazing Grace* (2006), which
teaches about William Wilberforce, the Slave Trade Act, 18th-century Parliament, etc.

Strategy:
1. Take the top-N concept QIDs by composite score
2. Query: which films/novels list any of these as P921 main subject?
3. Exclude works already in the corpus
4. Add surviving works flagged as `source = 'p921_expansion'`

In [28]:
P921_CACHE = os.path.join(CACHE_DIR, 'p921_expansion.csv')

# Use top-2000 concept QIDs by composite score as expansion seeds
EXPANSION_TOP_N = 2000
SITELINKS_MIN   = 15  # minimum sitelinks for expansion works (quality filter)

if 'composite_score' in composite.columns:
    top_concepts = (
        composite[composite['qid'].notna()]
        .nlargest(EXPANSION_TOP_N, 'composite_score')
    )
else:
    top_concepts = composite[composite['qid'].notna()].head(EXPANSION_TOP_N)

expansion_qids = top_concepts['qid'].dropna().astype(str).tolist()
expansion_qids = [q for q in expansion_qids if q.startswith('Q')]
print(f'Expansion seed QIDs: {len(expansion_qids):,}')


if os.path.exists(P921_CACHE):
    p921_raw = pd.read_csv(P921_CACHE)
    print(f'Loaded P921 cache: {len(p921_raw):,} rows')

else:
    # Batch into groups of 100 QIDs (SPARQL VALUES clause limit)
    BATCH_SIZE = 100
    batches = [expansion_qids[i:i+BATCH_SIZE]
               for i in range(0, len(expansion_qids), BATCH_SIZE)]

    all_p921 = []
    for b_idx, batch in enumerate(batches):
        values = ' '.join(f'wd:{q}' for q in batch)
        query = f"""
SELECT DISTINCT ?work ?workLabel ?workDescription ?workType ?sitelinks
                ?subject ?subjectLabel ?pubdate WHERE {{
  VALUES ?concept {{ {values} }}
  ?work wdt:P921 ?concept .
  ?work wdt:P31 ?workType .
  VALUES ?workType {{
    wd:Q11424    # film
    wd:Q7725634  # literary work
    wd:Q571      # book
    wd:Q8261     # novel
  }}
  ?work wikibase:sitelinks ?sitelinks .
  FILTER(?sitelinks >= {SITELINKS_MIN})
  ?work wdt:P921 ?subject .
  OPTIONAL {{ ?work wdt:P577 ?pubdate . }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
}}
ORDER BY DESC(?sitelinks)
"""
        if b_idx % 5 == 0:
            print(f'  P921 batch {b_idx+1}/{len(batches)}...', end=' ')
        result = run_sparql(query)
        if b_idx % 5 == 0:
            print(f'{len(result):,} rows')
        if len(result) > 0:
            all_p921.append(result)
        time.sleep(3)

    if all_p921:
        p921_raw = pd.concat(all_p921, ignore_index=True)
        p921_raw.to_csv(P921_CACHE, index=False)
        print(f'Saved P921 cache: {len(p921_raw):,} rows')
    else:
        p921_raw = pd.DataFrame()
        print('No P921 results returned.')

Expansion seed QIDs: 2,000
Loaded P921 cache: 844 rows


In [29]:
if len(p921_raw) > 0:
    p921 = p921_raw.copy()
    p921['qid']       = p921['work'].str.extract(r'(Q\d+)$')
    p921['sitelinks'] = pd.to_numeric(p921['sitelinks'], errors='coerce') \
                        if 'sitelinks' in p921.columns else np.nan
    p921['title']     = p921['workLabel'] if 'workLabel' in p921.columns else p921.get('work', '')
    p921['wd_desc']   = p921['workDescription'] if 'workDescription' in p921.columns else ''
    p921['pub_year']  = p921['pubdate'].str.extract(r'(\d{4})').astype(float) \
                        if 'pubdate' in p921.columns else np.nan

    type_map = {'Q11424': 'film', 'Q7725634': 'novel', 'Q571': 'novel', 'Q8261': 'novel'}
    if 'workType' in p921.columns:
        p921['medium'] = p921['workType'].str.extract(r'(Q\d+)$').iloc[:, 0].map(type_map)
    else:
        p921['medium'] = np.nan
    p921['medium'] = p921['medium'].fillna(
        p921['wd_desc'].str.lower().str.contains('film|movie', na=False).map(
            {True: 'film', False: 'novel'}
        )
    )

    p921_collapsed = p921.groupby('qid').agg(
        title=('title', 'first'),
        wd_desc=('wd_desc', 'first'),
        medium=('medium', 'first'),
        sitelinks=('sitelinks', 'max'),
        year=('pub_year', 'min'),
        p921_subjects=('subjectLabel', lambda x: '; '.join(sorted(set(x.dropna())))) \
                      if 'subjectLabel' in p921.columns else ('title', 'first'),
    ).reset_index()

    existing_qids_corpus = set(corpus['qid'].dropna())
    p921_new = p921_collapsed[~p921_collapsed['qid'].isin(existing_qids_corpus)].copy()
    p921_new['norm_title'] = p921_new['title'].apply(normalize)
    p921_new['source']     = 'p921_expansion'
    p921_new['list_count'] = 0
    start_id = len(corpus)
    p921_new['work_id'] = ['W{:05d}'.format(i + start_id) for i in range(len(p921_new))]
    print(f'P921 new works: {len(p921_new):,}')
else:
    p921_new = pd.DataFrame()
    print('No P921 results.')


P921 new works: 75


In [30]:
# Add P921 expansion works to corpus
if len(p921_new) > 0:
    # P921 expansion works have no list rankings — set importance from sitelinks only
    p921_new['pctile_sitelinks'] = pctile(p921_new['sitelinks'])
    p921_new['importance_score'] = p921_new['pctile_sitelinks']
    p921_new['n_importance_signals'] = 1
    p921_new['age_tier'] = 'adult'  # conservative default; refine in Notebook 12

    corpus = pd.concat([corpus, p921_new], ignore_index=True, sort=False)
    print(f'Corpus after P921 expansion: {len(corpus):,} works')
    print(f'  Films:  {(corpus["medium"] == "film").sum():,}')
    print(f'  Novels: {(corpus["medium"] == "novel").sum():,}')
else:
    print(f'Corpus unchanged: {len(corpus):,} works')

# Mark source for canonical-list works
corpus['source'] = corpus['source'].fillna('canonical_list')

Corpus after P921 expansion: 883 works
  Films:  444
  Novels: 439


---
## Section 10: Join to Concept Dataset

Check which works are already tracked as concepts in `cultural_salience_composite_v2.csv`.
Works that ARE concepts (e.g. *Schindler's List* appears in Jeopardy + crosswords) already
have a composite salience score — that score is one of our importance signals.

In [31]:
composite_qid_idx  = composite.set_index('qid') if 'qid' in composite.columns else None
composite_norm_idx = composite.set_index('norm')

def _scalar(val):
    if hasattr(val, 'iloc'):
        return float(val.iloc[0]) if len(val) > 0 else np.nan
    return float(val) if pd.notna(val) else np.nan

def get_concept_score(row):
    if composite_qid_idx is not None and pd.notna(row.get('qid')):
        if row['qid'] in composite_qid_idx.index and 'composite_score' in composite_qid_idx.columns:
            return _scalar(composite_qid_idx.loc[row['qid'], 'composite_score'])
    norm = normalize(row.get('title', ''))
    if norm in composite_norm_idx.index and 'composite_score' in composite_norm_idx.columns:
        return _scalar(composite_norm_idx.loc[norm, 'composite_score'])
    return np.nan

corpus['concept_dataset_score'] = corpus.apply(get_concept_score, axis=1)
corpus['in_concept_dataset'] = corpus['concept_dataset_score'].notna()
print(f'In concept dataset: {corpus["in_concept_dataset"].sum():,} / {len(corpus):,}')


In concept dataset: 564 / 883


---
## Section 11: Final Cleanup & Export

In [32]:
# ── Select and order final columns ─────────────────────────────────────────────
CORE_COLS = [
    'work_id', 'qid', 'title', 'norm_title', 'medium', 'year',
    'director', 'author', 'genre', 'runtime_min',
    'mpaa_rating', 'age_tier',
    'sitelinks', 'importance_score', 'n_importance_signals', 'list_count',
    'concept_dataset_score', 'in_concept_dataset',
    'cluster_id',
    'based_on_qid',
    'source',
    'wd_desc',
]

# Add per-list signal columns if present
list_signal_cols = [
    'afi_rank', 'ss_rank', 'imdb_rank', 'ebert_great', 'criterion',
    'ml_rank', 'time_100', 'pulitzer', 'booker', 'national_book_award',
]
list_signal_cols = [c for c in list_signal_cols if c in corpus.columns]

# P921 subjects for expansion works
extra_cols = [c for c in ['p921_subjects'] if c in corpus.columns]

final_cols = [c for c in CORE_COLS if c in corpus.columns] + list_signal_cols + extra_cols
corpus_out = corpus[final_cols].copy()

# Sort: films first, then novels, both by importance desc
corpus_out['_sort_medium'] = corpus_out['medium'].map({'film': 0, 'novel': 1}).fillna(2)
corpus_out = corpus_out.sort_values(
    ['_sort_medium', 'importance_score'], ascending=[True, False]
).drop(columns=['_sort_medium']).reset_index(drop=True)

print(f'Final corpus: {len(corpus_out):,} works')
print(f'  Films:  {(corpus_out["medium"]=="film").sum():,}')
print(f'  Novels: {(corpus_out["medium"]=="novel").sum():,}')
print(f'\nColumn list ({len(final_cols)} cols): {final_cols}')

Final corpus: 883 works
  Films:  444
  Novels: 439

Column list (33 cols): ['work_id', 'qid', 'title', 'norm_title', 'medium', 'year', 'director', 'author', 'genre', 'runtime_min', 'mpaa_rating', 'age_tier', 'sitelinks', 'importance_score', 'n_importance_signals', 'list_count', 'concept_dataset_score', 'in_concept_dataset', 'cluster_id', 'based_on_qid', 'source', 'wd_desc', 'afi_rank', 'ss_rank', 'imdb_rank', 'ebert_great', 'criterion', 'ml_rank', 'time_100', 'pulitzer', 'booker', 'national_book_award', 'p921_subjects']


In [33]:
# ── Export works_corpus.csv ────────────────────────────────────────────────────
corpus_path = os.path.join(DATA_DIR, 'works_corpus.csv')
corpus_out.to_csv(corpus_path, index=False)
print(f'Saved {len(corpus_out):,} works to {corpus_path}')

# ── Export work_clusters.csv ───────────────────────────────────────────────────
# Update clusters_df to reflect any new P921 works added to clusters
# (Re-run cluster logic on full corpus)
qid_to_work_id_full = {}
for _, row in corpus_out.iterrows():
    if pd.notna(row.get('qid')):
        qid_to_work_id_full[row['qid']] = row['work_id']

clusters_path = os.path.join(DATA_DIR, 'work_clusters.csv')
clusters_df.to_csv(clusters_path, index=False)
print(f'Saved {len(clusters_df):,} clusters to {clusters_path}')
print(f'  Adaptation clusters (novel+film): {len(adaptation_clusters):,}')

Saved 883 works to ../data\works_corpus.csv
Saved 803 clusters to ../data\work_clusters.csv
  Adaptation clusters (novel+film): 4


In [34]:
# ── Summary ────────────────────────────────────────────────────────────────────
print('=' * 65)
print('NOTEBOOK 09: WORKS CORPUS SUMMARY')
print('=' * 65)

print(f'\nWorks corpus: {len(corpus_out):,} total')
print(f'  Films:       {(corpus_out["medium"]=="film").sum():,}')
print(f'  Novels:      {(corpus_out["medium"]=="novel").sum():,}')

print(f'\nSources:')
print(corpus_out['source'].value_counts().to_string())

print(f'\nAge tier distribution:')
print(corpus_out.groupby(['medium','age_tier']).size().to_string())

print(f'\nClusters: {len(clusters_df):,} total')
print(f'  Adaptation clusters (novel + film): {len(adaptation_clusters):,}')

print(f'\nWikidata coverage:')
print(f'  QID-linked: {corpus_out["qid"].notna().sum():,} / {len(corpus_out):,}')
print(f'  In concept dataset: {corpus_out["in_concept_dataset"].sum():,}')

print(f'\nImportance score stats:')
print(corpus_out['importance_score'].describe().round(1).to_string())

print(f'\nTop 10 films by importance:')
print(
    corpus_out[corpus_out['medium']=='film']
    .nlargest(10, 'importance_score')
    [['title','year','importance_score','list_count']]
    .to_string(index=False)
)
print(f'\nTop 10 novels by importance:')
print(
    corpus_out[corpus_out['medium']=='novel']
    .nlargest(10, 'importance_score')
    [['title','year','importance_score','list_count']]
    .to_string(index=False)
)

print(f'\nOutputs:')
print(f'  data/works_corpus.csv   ({len(corpus_out):,} rows)')
print(f'  data/work_clusters.csv  ({len(clusters_df):,} rows)')

NOTEBOOK 09: WORKS CORPUS SUMMARY

Works corpus: 883 total
  Films:       444
  Novels:      439

Sources:
source
canonical_list    808
p921_expansion     75

Age tier distribution:
medium  age_tier
film    adult       391
        child        13
        middle       26
        ya           14
novel   adult       438
        ya            1

Clusters: 803 total
  Adaptation clusters (novel + film): 4

Wikidata coverage:
  QID-linked: 347 / 883
  In concept dataset: 564

Importance score stats:
count    840.0
mean      68.0
std       27.8
min        0.8
25%       48.0
50%       69.7
75%      100.0
max      100.0

Top 10 films by importance:
                    title    year  importance_score  list_count
                    Ikiru  1952.0             100.0           2
                  Yojimbo  1961.0             100.0           2
              Tokyo Story  1953.0             100.0           2
              Late Spring  1949.0             100.0           2
                 Amarcord  1973.